# LeetCode 763 — Partition Labels bằng Python

Link đề: https://leetcode.com/problems/partition-labels/ · Độ khó: **Medium** · Chủ đề: Greedy (tham lam), Hash Table, Two Pointers, String

Skeleton LeetCode:

```python
class Solution:
    def partitionLabels(self, s: str) -> List[int]:
        ...
```

Ghi chú: bài là một ví dụ rất đẹp của **tham lam + "vị trí xuất hiện cuối cùng"**. Notebook đi từ cách thử từng đoạn (chậm), qua bản ghi nhớ vị trí cuối (greedy chuẩn), rồi nhìn bài toán như **gộp khoảng** (Merge Intervals), như **đếm ngược số lần xuất hiện**, và như **quét đường thẳng (sweep line) với mảng hiệu**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cắt chuỗi `s` thành **nhiều đoạn nhất có thể** sao cho mỗi chữ cái chỉ xuất hiện trong **một** đoạn; trả về độ dài các đoạn theo thứ tự.
2. Nắm các công cụ Python cần thiết: `enumerate`, dict comprehension (ghi đè khóa trùng), `max`, `set` và giao tập hợp `&`, `collections.Counter`, `sorted` với tuple, `itertools.accumulate`/mảng hiệu.
3. Hiểu trực giác: nếu một chữ cái xuất hiện ở nhiều chỗ thì mọi thứ **từ lần đầu đến lần cuối** của nó phải nằm chung một đoạn, và các ràng buộc đó **lan truyền** (đoạn mở rộng khi gặp chữ có lần cuối xa hơn).
4. Đi qua 5 cách: thử từng đoạn bằng tập hợp, tham lam với `last[c]`, gộp khoảng `[first, last]`, đếm ngược số lần xuất hiện còn lại, mảng hiệu (sweep line); đối chiếu bằng oracle thử mọi tập điểm cắt.
5. Tránh các lỗi kinh điển: gán `mx = last[c]` thay vì `max`, đóng đoạn khi `i == last[c]` thay vì `i == mx`, dựng `last` bằng lần xuất hiện đầu, quên đặt lại đầu đoạn, lệch 1 khi tính độ dài.

## 2. Đề bài & ví dụ

Cho chuỗi `s`. Ta muốn chia `s` thành **càng nhiều phần càng tốt** sao cho **mỗi chữ cái xuất hiện trong nhiều nhất một phần**. Ví dụ `"ababcc"` có thể chia thành `["abab", "cc"]`; còn `["aba", "bcc"]` hoặc `["ab", "ab", "cc"]` là không hợp lệ.

Lưu ý: ghép các phần theo thứ tự phải ra đúng `s`. Trả về **danh sách độ dài** của các phần.

**Ví dụ 1**

- Input: `s = "ababcbacadefegdehijhklij"`
- Output: `[9,7,8]`
- Giải thích: các phần là `"ababcbaca"`, `"defegde"`, `"hijhklij"`. Cách chia `"ababcbacadefegde"`, `"hijhklij"` không đúng vì ít phần hơn.

**Ví dụ 2**

- Input: `s = "eccbbbbdec"`
- Output: `[10]`

**Ràng buộc:**

- `1 <= s.length <= 500`
- `s` chỉ gồm chữ cái thường tiếng Anh.

Một điều quan trọng: cách chia nhiều phần nhất là **duy nhất** (ta sẽ thấy vì sao), nên bài có đáp án xác định để so sánh bằng `==`.

In [1]:
from typing import List
from collections import Counter
from itertools import accumulate
import random, time

examples = [
    ("ababcbacadefegdehijhklij", [9, 7, 8]),
    ("eccbbbbdec", [10]),
]
for s, expected in examples:
    print(f"s={s!r:28} len={len(s):2} kỳ vọng={expected}")

s='ababcbacadefegdehijhklij'   len=24 kỳ vọng=[9, 7, 8]
s='eccbbbbdec'                 len=10 kỳ vọng=[10]


## 3. Kiến thức nền

### 3.1. `enumerate` và dict comprehension: ghi đè khóa trùng

`enumerate(s)` cho `(chỉ số, ký tự)`. Trong dict comprehension `{c: i for i, c in enumerate(s)}`, khi một ký tự xuất hiện nhiều lần, **lần sau ghi đè lần trước**, nên cuối cùng `d[c]` là **chỉ số xuất hiện cuối cùng** của `c`. Đây là cách dựng bảng `last` trong một dòng.

In [2]:
s = "abaccb"
print(list(enumerate(s)))
last = {c: i for i, c in enumerate(s)}
print("last =", last)                      # lần sau ghi đè lần trước
first = {}
for i, c in enumerate(s):
    first.setdefault(c, i)                 # setdefault: chỉ đặt nếu chưa có -> lần đầu
print("first =", first)

[(0, 'a'), (1, 'b'), (2, 'a'), (3, 'c'), (4, 'c'), (5, 'b')]
last = {'a': 2, 'b': 5, 'c': 4}
first = {'a': 0, 'b': 1, 'c': 3}


### 3.2. `max` chạy dần và "đường chân trời"

`mx = max(mx, x)` giữ **giá trị lớn nhất đã gặp**, như một đường chân trời chỉ tăng. Trong bài này `mx` là vị trí xa nhất mà đoạn hiện tại **bắt buộc** phải vươn tới.

In [3]:
xs = [3, 1, 4, 1, 5, 9, 2, 6]
mx = 0
for x in xs:
    mx = max(mx, x)
    print(x, "->", mx, end=" | ")
print()
print("viết sai: mx = x (ghi đè) ->", [x for x in xs])

3 -> 3 | 1 -> 3 | 4 -> 4 | 1 -> 4 | 5 -> 5 | 9 -> 9 | 2 -> 9 | 6 -> 9 | 
viết sai: mx = x (ghi đè) -> [3, 1, 4, 1, 5, 9, 2, 6]


### 3.3. Tập hợp `set` và giao `&`

`set(s)` là tập các ký tự khác nhau. `a & b` là **giao** (phần chung). `not (a & b)` đúng khi hai tập **rời nhau**. Dùng để hỏi "chữ nào trong đoạn này còn xuất hiện ở phần sau không?".

In [4]:
s = "ababcc"
left, right = set(s[:4]), set(s[4:])
print(left, right, "giao:", left & right, "rời nhau?", not (left & right))
left, right = set(s[:3]), set(s[3:])
print(left, right, "giao:", left & right, "rời nhau?", not (left & right))

{'b', 'a'} {'c'} giao: set() rời nhau? True
{'b', 'a'} {'b', 'c'} giao: {'b'} rời nhau? False


### 3.4. `Counter` và đếm ngược

`Counter(s)` đếm số lần mỗi ký tự xuất hiện. Ta có thể **giảm** dần khi duyệt qua: khi một ký tự về `0` nghĩa là đã dùng hết lần xuất hiện của nó (dùng ở Cách 4).

In [5]:
cnt = Counter("abaccb")
print(cnt)
for c in "abaccb":
    cnt[c] -= 1
    print(c, dict(cnt))

Counter({'a': 2, 'b': 2, 'c': 2})
a {'a': 1, 'b': 2, 'c': 2}
b {'a': 1, 'b': 1, 'c': 2}
a {'a': 0, 'b': 1, 'c': 2}
c {'a': 0, 'b': 1, 'c': 1}
c {'a': 0, 'b': 1, 'c': 0}
b {'a': 0, 'b': 0, 'c': 0}


### 3.5. Sắp xếp danh sách tuple và mảng hiệu (difference array)

- `sorted([(3, 5), (0, 2)])` sắp theo phần tử đầu rồi phần tử sau: `[(0, 2), (3, 5)]`.
- **Mảng hiệu**: để cộng `+1` cho cả đoạn `[l, r)` ta ghi `diff[l] += 1`, `diff[r] -= 1`; tổng tích lũy (`accumulate`) cho giá trị tại mỗi vị trí. Cách này đánh dấu nhiều đoạn chỉ với `O(1)` mỗi đoạn.

In [6]:
print(sorted([(3, 5), (0, 2), (1, 1)]))

n = 8
diff = [0] * (n + 1)
for l, r in [(1, 4), (2, 6)]:          # cộng 1 cho các vị trí l..r-1
    diff[l] += 1
    diff[r] -= 1
print("diff  :", diff)
print("cover :", list(accumulate(diff))[:n])

[(0, 2), (1, 1), (3, 5)]
diff  : [0, 1, 1, 0, -1, 0, -1, 0, 0]
cover : [0, 1, 2, 2, 1, 1, 0, 0]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Xét `s = "ababcbacadefegdehijhklij"`. Ghi lại **chỉ số xuất hiện cuối cùng** của từng chữ:

- `a` -> 8, `b` -> 5, `c` -> 7, `d` -> 14, `e` -> 15, `f` -> 11, `g` -> 13, `h` -> 19, `i` -> 22, `j` -> 23, `k` -> 20, `l` -> 21.

Bắt đầu đoạn đầu tiên ở chỉ số 0. Ký tự `a` xuất hiện lần cuối ở 8 nên **đoạn phải kéo dài tới ít nhất 8**. Đi dọc theo đoạn: gặp `b` (cuối ở 5, đã nằm trong), `c` (cuối ở 7, trong), ... Tới chỉ số 8 thì `mx = 8`: không chữ nào trong đoạn có lần cuối xa hơn 8, ta **cắt** ở 8 -> đoạn dài 9. Làm tiếp ở chỉ số 9: `d` cuối ở 14, `e` cuối ở 15 -> kéo tới 15 -> đoạn dài 7. Cuối cùng `hijhklij` -> đoạn dài 8.

Quy tắc: **đoạn hiện tại phải vươn tới `max(last[c])` của mọi chữ `c` đã nằm trong nó; cắt ngay khi chỉ số hiện tại chạm tới mức đó**.

In [7]:
s = "ababcbacadefegdehijhklij"
last = {c: i for i, c in enumerate(s)}
print(sorted(last.items()))
start = mx = 0
for i, c in enumerate(s):
    mx = max(mx, last[c])
    note = "  <- cắt ở đây" if i == mx else ""
    print(f"i={i:2} c={c} last[c]={last[c]:2} mx={mx:2}{note}")
    if i == mx:
        print(f"    đoạn [{start}..{i}] = {s[start:i + 1]!r}, dài {i - start + 1}")
        start = i + 1

[('a', 8), ('b', 5), ('c', 7), ('d', 14), ('e', 15), ('f', 11), ('g', 13), ('h', 19), ('i', 22), ('j', 23), ('k', 20), ('l', 21)]
i= 0 c=a last[c]= 8 mx= 8
i= 1 c=b last[c]= 5 mx= 8
i= 2 c=a last[c]= 8 mx= 8
i= 3 c=b last[c]= 5 mx= 8
i= 4 c=c last[c]= 7 mx= 8
i= 5 c=b last[c]= 5 mx= 8
i= 6 c=a last[c]= 8 mx= 8
i= 7 c=c last[c]= 7 mx= 8
i= 8 c=a last[c]= 8 mx= 8  <- cắt ở đây
    đoạn [0..8] = 'ababcbaca', dài 9
i= 9 c=d last[c]=14 mx=14
i=10 c=e last[c]=15 mx=15
i=11 c=f last[c]=11 mx=15
i=12 c=e last[c]=15 mx=15
i=13 c=g last[c]=13 mx=15
i=14 c=d last[c]=14 mx=15
i=15 c=e last[c]=15 mx=15  <- cắt ở đây
    đoạn [9..15] = 'defegde', dài 7
i=16 c=h last[c]=19 mx=19
i=17 c=i last[c]=22 mx=22
i=18 c=j last[c]=23 mx=23
i=19 c=h last[c]=19 mx=23
i=20 c=k last[c]=20 mx=23
i=21 c=l last[c]=21 mx=23
i=22 c=i last[c]=22 mx=23
i=23 c=j last[c]=23 mx=23  <- cắt ở đây
    đoạn [16..23] = 'hijhklij', dài 8


## 5. Cách 1 — Brute Force: thử kéo dài đoạn cho tới khi "đóng"

**Ý tưởng.** Ở vị trí đầu đoạn `j`, thử độ dài tăng dần: đoạn `s[j..i]` hợp lệ ("đóng") khi **không chữ nào trong nó xuất hiện ở phần còn lại** `s[i+1:]`. Gặp đoạn đóng đầu tiên thì cắt (cắt sớm nhất = nhiều đoạn nhất), rồi làm tiếp từ `i + 1`.

Mỗi lần kiểm tra ta dựng hai `set` rồi xét giao: dễ hiểu, không cần nghĩ tới `last`.

In [8]:
def partition_brute(s: str) -> List[int]:
    n = len(s)
    ans = []
    j = 0
    while j < n:
        i = j
        while True:
            inside = set(s[j:i + 1])
            outside = set(s[i + 1:])
            if not (inside & outside):       # không chữ nào lọt ra ngoài -> đoạn đóng
                break
            i += 1
        ans.append(i - j + 1)
        j = i + 1
    return ans

for s, expected in examples:
    got = partition_brute(s)
    print(s, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

ababcbacadefegdehijhklij -> [9, 7, 8] OK
eccbbbbdec -> [10] OK


**Oracle độc lập.** Với chuỗi ngắn, ta có thể thử **mọi tập điểm cắt** (có `n - 1` vị trí giữa các ký tự, mỗi vị trí cắt hoặc không: `2^(n-1)` cách), giữ cách hợp lệ có **nhiều phần nhất**. Nó đúng theo định nghĩa, không dùng bất kỳ ý tưởng nào của các cách giải.

In [9]:
def oracle(s: str) -> List[int]:
    n = len(s)
    distinct = len(set(s))
    best = None
    for mask in range(1 << (n - 1)):
        parts, prev = [], 0
        for i in range(n - 1):
            if mask >> i & 1:
                parts.append(s[prev:i + 1])
                prev = i + 1
        parts.append(s[prev:])
        # hợp lệ <=> các tập chữ cái của các phần đôi một rời nhau <=> tổng kích thước tập = số chữ khác nhau
        if sum(len(set(p)) for p in parts) == distinct:
            if best is None or len(parts) > len(best):
                best = parts
    return [len(p) for p in best]

for s, expected in examples[:2]:
    if len(s) <= 12:
        assert oracle(s) == expected
print(oracle("ababcc"), oracle("eccbbbbdec"), oracle("abcd"))

[4, 2] [10] [1, 1, 1, 1]


**Độ phức tạp và nút thắt.** Với mỗi vị trí `i` ta dựng `set(s[j:i+1])` và `set(s[i+1:])`, mỗi lần `O(n)`; có tối đa `n` giá trị `i` cho toàn bộ chuỗi (con trỏ `i` không quay lại), nên tổng `O(n^2)` với hằng số dựng set. Bộ nhớ `O(1)` (tập tối đa 26 phần tử) hoặc `O(n)` cho lát cắt. Với `n <= 500` thì vẫn chạy được, nhưng **mỗi lần kiểm tra lại quét toàn chuỗi** dù thông tin "chữ nào còn xuất hiện về sau" không đổi. Nút thắt: tính đi tính lại `outside`.

## 6. Ý tưởng thuật toán / bước đột phá

Thay vì hỏi "chữ nào còn xuất hiện về sau" mỗi lần, ta **tính trước** một lần: `last[c]` = chỉ số xuất hiện cuối cùng của `c`. Khi đó:

- Nếu `c` nằm trong đoạn hiện tại thì đoạn phải kéo dài tới **ít nhất** `last[c]`.
- `mx = max(last[c])` trên mọi `c` đã duyệt trong đoạn là "đường chân trời". Khi chỉ số `i` chạm `mx`, mọi chữ trong đoạn đã xuất hiện lần cuối trong đoạn -> **cắt được**.
- Cắt **ngay lúc đầu tiên** có thể là cách nhiều đoạn nhất (cắt sớm hơn thì vi phạm; cắt muộn hơn thì ít đoạn hơn). Đó là bản chất tham lam.

Ba cách nhìn khác nhau nhưng tương đương:

| Cách | Ý tưởng |
|---|---|
| 2 | Tham lam với `last[c]` và `mx` |
| 3 | Mỗi chữ là một khoảng `[first, last]`; **gộp các khoảng giao nhau**, mỗi khoảng gộp là một đoạn |
| 4 | Đếm ngược số lần xuất hiện còn lại; cắt khi **không còn chữ nào đang "mở"** |
| 5 | Mảng hiệu: đếm "số chữ đang bắc cầu qua khe giữa `i` và `i+1`"; cắt khi bằng 0 |

Cả bốn đều `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam với bảng `last`

**Giải thích từng đoạn.**

- `last = {c: i for i, c in enumerate(s)}`: vị trí cuối của mỗi chữ.
- `j` = đầu đoạn hiện tại, `mx` = đường chân trời.
- Với mỗi `i`: `mx = max(mx, last[s[i]])`. Nếu `mx == i`, đoạn `[j..i]` đóng: thêm độ dài `i - j + 1`, đặt `j = i + 1`.

In [10]:
def partition_greedy(s: str) -> List[int]:
    last = {c: i for i, c in enumerate(s)}
    mx = j = 0
    ans = []
    for i, c in enumerate(s):
        mx = max(mx, last[c])
        if mx == i:
            ans.append(i - j + 1)
            j = i + 1
    return ans

for s, expected in examples:
    assert partition_greedy(s) == expected
print("greedy OK")

greedy OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def partitionLabels(self, s: str) -> List[int]:
        last = {c: i for i, c in enumerate(s)}
        mx = j = 0
        ans = []
        for i, c in enumerate(s):
            mx = max(mx, last[c])
            if mx == i:
                ans.append(i - j + 1)
                j = i + 1
        return ans

sol_greedy = Solution().partitionLabels
print([sol_greedy(s) for s, _ in examples])

[[9, 7, 8], [10]]


**Độ phức tạp (Cách 2):** thời gian `O(n)` (hai lượt qua chuỗi), bộ nhớ `O(|Σ|) = O(26)` cho `last`.

---

### 7.2. Cách 3 — Gộp khoảng `[first, last]` (Merge Intervals)

**Ý tưởng khác.** Mỗi chữ `c` xuất hiện ít nhất một lần tạo một **khoảng** `[first[c], last[c]]` mà mọi vị trí trong đó phải nằm chung một đoạn. Hai khoảng **giao nhau** buộc phải chung đoạn. Vậy các đoạn đáp án chính là các **khoảng sau khi gộp** (đúng bài Merge Intervals).

**Giải thích từng đoạn.**

- Dựng `first` và `last`.
- Sắp các khoảng theo `first`.
- Duyệt: nếu khoảng mới bắt đầu `<=` đầu mút phải của khoảng gộp gần nhất thì gộp (`max` hai đầu mút phải); ngược lại mở khoảng gộp mới.
- Mỗi chỉ số thuộc khoảng của chữ ở đó nên các khoảng gộp **lát kín** chuỗi; độ dài mỗi khoảng là một phần tử đáp án.

In [12]:
def partition_intervals(s: str) -> List[int]:
    first, last = {}, {}
    for i, c in enumerate(s):
        first.setdefault(c, i)
        last[c] = i
    intervals = sorted((first[c], last[c]) for c in first)
    merged = []
    for a, b in intervals:
        if merged and a <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    return [b - a + 1 for a, b in merged]

for s, expected in examples:
    assert partition_intervals(s) == expected
print("intervals OK")

intervals OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def partitionLabels(self, s: str) -> List[int]:
        first, last = {}, {}
        for i, c in enumerate(s):
            first.setdefault(c, i)
            last[c] = i
        merged = []
        for a, b in sorted((first[c], last[c]) for c in first):
            if merged and a <= merged[-1][1]:
                merged[-1][1] = max(merged[-1][1], b)
            else:
                merged.append([a, b])
        return [b - a + 1 for a, b in merged]

sol_intervals = Solution().partitionLabels
print([sol_intervals(s) for s, _ in examples])

[[9, 7, 8], [10]]


**Độ phức tạp (Cách 3):** thời gian `O(n + |Σ| log |Σ|)` (sắp xếp tối đa 26 khoảng), bộ nhớ `O(|Σ|)`.

---

### 7.3. Cách 4 — Đếm ngược số lần xuất hiện còn lại

**Ý tưởng.** Duy trì `remaining[c]` = số lần `c` **còn chưa duyệt tới** và tập `opened` các chữ đã xuất hiện trong đoạn hiện tại mà **vẫn còn lần xuất hiện sắp tới**. Khi `opened` rỗng nghĩa là mọi chữ trong đoạn đã "dùng hết" -> cắt.

**Giải thích từng đoạn.**

- `remaining = Counter(s)`.
- Với mỗi `c` ở chỉ số `i`: thêm `c` vào `opened`, giảm `remaining[c]`; nếu `remaining[c] == 0` thì bỏ `c` khỏi `opened`.
- Nếu `opened` rỗng: đóng đoạn.

In [14]:
def partition_countdown(s: str) -> List[int]:
    remaining = Counter(s)
    opened = set()
    start = 0
    ans = []
    for i, c in enumerate(s):
        opened.add(c)
        remaining[c] -= 1
        if remaining[c] == 0:
            opened.discard(c)
        if not opened:
            ans.append(i - start + 1)
            start = i + 1
    return ans

for s, expected in examples:
    assert partition_countdown(s) == expected
print("countdown OK")

countdown OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def partitionLabels(self, s: str) -> List[int]:
        from collections import Counter
        remaining = Counter(s)
        opened = set()
        start = 0
        ans = []
        for i, c in enumerate(s):
            opened.add(c)
            remaining[c] -= 1
            if remaining[c] == 0:
                opened.discard(c)
            if not opened:
                ans.append(i - start + 1)
                start = i + 1
        return ans

sol_countdown = Solution().partitionLabels
print([sol_countdown(s) for s, _ in examples])

[[9, 7, 8], [10]]


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(|Σ|)`.

---

### 7.4. Cách 5 — Mảng hiệu (sweep line)

**Ý tưởng.** Xét "khe" ngay sau chỉ số `i` (giữa `i` và `i + 1`). Khe đó **bị chặn** nếu có chữ `c` với `first[c] <= i < last[c]` (chữ đó bắc cầu qua khe). Số chữ bắc cầu qua khe `i` là `cover[i]`. **Cắt được** ở khe `i` khi `cover[i] == 0`.

**Giải thích từng đoạn.**

- Với mỗi chữ: `diff[first] += 1`, `diff[last] -= 1` (cộng 1 cho chỉ số `first..last-1`).
- `cover[i]` = tổng tích lũy của `diff` tới `i`.
- Khi `cover[i] == 0` thì đóng đoạn tại `i`. Chữ chỉ xuất hiện một lần có `first == last`, `+1` và `-1` triệt tiêu nhau.

In [16]:
def partition_diff(s: str) -> List[int]:
    n = len(s)
    first, last = {}, {}
    for i, c in enumerate(s):
        first.setdefault(c, i)
        last[c] = i
    diff = [0] * (n + 1)
    for c in first:
        diff[first[c]] += 1
        diff[last[c]] -= 1
    ans = []
    cover = 0
    start = 0
    for i in range(n):
        cover += diff[i]
        if cover == 0:
            ans.append(i - start + 1)
            start = i + 1
    return ans

for s, expected in examples:
    assert partition_diff(s) == expected
print("diff array OK")

diff array OK


**Code nộp LeetCode (Cách 5):**

In [17]:
class Solution:
    def partitionLabels(self, s: str) -> List[int]:
        n = len(s)
        first, last = {}, {}
        for i, c in enumerate(s):
            first.setdefault(c, i)
            last[c] = i
        diff = [0] * (n + 1)
        for c in first:
            diff[first[c]] += 1
            diff[last[c]] -= 1
        ans = []
        cover = start = 0
        for i in range(n):
            cover += diff[i]
            if cover == 0:
                ans.append(i - start + 1)
                start = i + 1
        return ans

sol_diff = Solution().partitionLabels
print([sol_diff(s) for s, _ in examples])

[[9, 7, 8], [10]]


**Độ phức tạp (Cách 5):** thời gian `O(n)`, bộ nhớ `O(n)` cho mảng hiệu (cộng `O(|Σ|)`).

## 8. TRACE từng bước

### 8.1. Cách 2 trên `"ababcbacadefegdehijhklij"`

Bảng trạng thái (chỉ các dòng "đổi `mx`" hoặc "cắt"; xem hàm `trace_greedy` để thấy mọi dòng):

| `i` | ký tự | `last[c]` | `mx` sau | hành động |
|---|---|---|---|---|
| 0 | a | 8 | 8 | |
| 1 | b | 5 | 8 | |
| 2..7 | ... | <= 8 | 8 | |
| 8 | a | 8 | 8 | `mx == i` -> cắt, đoạn dài 9 |
| 9 | d | 14 | 14 | |
| 10 | e | 15 | 15 | |
| 15 | e | 15 | 15 | `mx == i` -> cắt, đoạn dài 7 |
| 16 | h | 19 | 19 | |
| 17 | i | 22 | 22 | |
| 18 | j | 23 | 23 | |
| 23 | j | 23 | 23 | `mx == i` -> cắt, đoạn dài 8 |

In [18]:
def trace_greedy(s: str) -> List[int]:
    last = {c: i for i, c in enumerate(s)}
    mx = j = 0
    ans = []
    for i, c in enumerate(s):
        old = mx
        mx = max(mx, last[c])
        note = ""
        if mx != old:
            note += f" (mx tăng {old} -> {mx})"
        if mx == i:
            ans.append(i - j + 1)
            note += f"  CẮT: đoạn {s[j:i + 1]!r} dài {i - j + 1}"
            j = i + 1
        print(f"i={i:2} c={c} last={last[c]:2} mx={mx:2}{note}")
    return ans

assert trace_greedy("ababcbacadefegdehijhklij") == [9, 7, 8]

i= 0 c=a last= 8 mx= 8 (mx tăng 0 -> 8)
i= 1 c=b last= 5 mx= 8
i= 2 c=a last= 8 mx= 8
i= 3 c=b last= 5 mx= 8
i= 4 c=c last= 7 mx= 8
i= 5 c=b last= 5 mx= 8
i= 6 c=a last= 8 mx= 8
i= 7 c=c last= 7 mx= 8
i= 8 c=a last= 8 mx= 8  CẮT: đoạn 'ababcbaca' dài 9
i= 9 c=d last=14 mx=14 (mx tăng 8 -> 14)
i=10 c=e last=15 mx=15 (mx tăng 14 -> 15)
i=11 c=f last=11 mx=15
i=12 c=e last=15 mx=15
i=13 c=g last=13 mx=15
i=14 c=d last=14 mx=15
i=15 c=e last=15 mx=15  CẮT: đoạn 'defegde' dài 7
i=16 c=h last=19 mx=19 (mx tăng 15 -> 19)
i=17 c=i last=22 mx=22 (mx tăng 19 -> 22)
i=18 c=j last=23 mx=23 (mx tăng 22 -> 23)
i=19 c=h last=19 mx=23
i=20 c=k last=20 mx=23
i=21 c=l last=21 mx=23
i=22 c=i last=22 mx=23
i=23 c=j last=23 mx=23  CẮT: đoạn 'hijhklij' dài 8


### 8.2. Cách 3 (gộp khoảng) cho `"eccbbbbdec"`

`first/last`: `e: [0, 8]`, `c: [1, 9]`, `b: [3, 6]`, `d: [7, 7]`. Sắp theo đầu mút trái: `(0,8) (1,9) (3,6) (7,7)`. Gộp: `[0,8]`; `(1,9)` giao nhau -> `[0,9]`; `(3,6)` nằm trong; `(7,7)` nằm trong. Một khoảng gộp `[0, 9]` -> `[10]`.

In [19]:
def trace_intervals(s: str) -> List[int]:
    first, last = {}, {}
    for i, c in enumerate(s):
        first.setdefault(c, i)
        last[c] = i
    intervals = sorted((first[c], last[c], c) for c in first)
    print("khoảng theo từng chữ:", [(c, a, b) for a, b, c in intervals])
    merged = []
    for a, b, c in intervals:
        if merged and a <= merged[-1][1]:
            old = list(merged[-1])
            merged[-1][1] = max(merged[-1][1], b)
            print(f"  {c}: [{a},{b}] giao với {old} -> gộp thành {merged[-1]}")
        else:
            merged.append([a, b])
            print(f"  {c}: [{a},{b}] không giao -> khoảng mới {merged[-1]}")
    return [b - a + 1 for a, b in merged]

assert trace_intervals("eccbbbbdec") == [10]
print()
assert trace_intervals("ababcc") == [4, 2]

khoảng theo từng chữ: [('e', 0, 8), ('c', 1, 9), ('b', 3, 6), ('d', 7, 7)]
  e: [0,8] không giao -> khoảng mới [0, 8]
  c: [1,9] giao với [0, 8] -> gộp thành [0, 9]
  b: [3,6] giao với [0, 9] -> gộp thành [0, 9]
  d: [7,7] giao với [0, 9] -> gộp thành [0, 9]

khoảng theo từng chữ: [('a', 0, 2), ('b', 1, 3), ('c', 4, 5)]
  a: [0,2] không giao -> khoảng mới [0, 2]
  b: [1,3] giao với [0, 2] -> gộp thành [0, 3]
  c: [4,5] không giao -> khoảng mới [4, 5]


### 8.3. Cách 5 (mảng hiệu) cho `"ababcc"`

`first/last`: `a: [0, 2]`, `b: [1, 3]`, `c: [4, 5]`. `diff`: `+1` ở 0, `-1` ở 2; `+1` ở 1, `-1` ở 3; `+1` ở 4, `-1` ở 5. `cover` theo `i = 0..5`: `1, 2, 1, 0, 1, 0`. Hai lần `cover == 0` ở `i = 3` và `i = 5` -> các đoạn dài `4` và `2`.

In [20]:
def trace_diff(s: str) -> List[int]:
    n = len(s)
    first, last = {}, {}
    for i, c in enumerate(s):
        first.setdefault(c, i)
        last[c] = i
    diff = [0] * (n + 1)
    for c in first:
        diff[first[c]] += 1
        diff[last[c]] -= 1
    print("diff  =", diff)
    ans, cover, start = [], 0, 0
    for i in range(n):
        cover += diff[i]
        note = ""
        if cover == 0:
            ans.append(i - start + 1)
            note = f"  CẮT, đoạn {s[start:i + 1]!r}"
            start = i + 1
        print(f"i={i} ({s[i]}) cover={cover}{note}")
    return ans

assert trace_diff("ababcc") == [4, 2]

diff  = [1, 1, -1, -1, 1, -1, 0]
i=0 (a) cover=1
i=1 (b) cover=2
i=2 (a) cover=1
i=3 (b) cover=0  CẮT, đoạn 'abab'
i=4 (c) cover=1
i=5 (c) cover=0  CẮT, đoạn 'cc'


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (tham lam với `last`): ngắn, nhanh, dễ giải thích và là lời giải chuẩn.

```python
class Solution:
    def partitionLabels(self, s: str) -> List[int]:
        last = {c: i for i, c in enumerate(s)}
        mx = j = 0
        ans = []
        for i, c in enumerate(s):
            mx = max(mx, last[c])
            if mx == i:
                ans.append(i - j + 1)
                j = i + 1
        return ans
```

In [21]:
class Solution:
    def partitionLabels(self, s: str) -> List[int]:
        last = {c: i for i, c in enumerate(s)}
        mx = j = 0
        ans = []
        for i, c in enumerate(s):
            mx = max(mx, last[c])
            if mx == i:
                ans.append(i - j + 1)
                j = i + 1
        return ans

sol = Solution()
for s, expected in examples:
    got = sol.partitionLabels(s)
    print(s, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

ababcbacadefegdehijhklij -> [9, 7, 8] OK
eccbbbbdec -> [10] OK


## 10. Vì sao đúng

**(a) Mỗi đoạn hợp lệ.** Khi cắt ở `i` với `mx == i`: mọi chữ `c` đã duyệt trong đoạn `[j..i]` có `last[c] <= mx = i`, tức không còn xuất hiện sau `i`; còn các chữ nằm ở các đoạn trước đã được "đóng" từ trước (không xuất hiện sau chỗ cắt trước). Vậy một chữ không thể rơi vào hai đoạn khác nhau.

**(b) Không thể có nhiều đoạn hơn (tối ưu).** Ta chứng minh mọi phép cắt hợp lệ chỉ có thể xảy ra tại các chỗ mà thuật toán cắt. Giả sử cắt hợp lệ giữa `i` và `i + 1`. Khi đó mọi chữ `c` ở vị trí `<= i` có `last[c] <= i` (nếu không `c` xuất hiện ở cả hai bên khe). Do đó `max(last[c])` trên đoạn `[0..i]` bằng `i`... tương ứng đúng với điều kiện cắt của thuật toán trên phần đã duyệt (ở mức toàn chuỗi, không chỉ trong đoạn hiện tại). Ngược lại thuật toán cắt tại `i` khi `mx == i`, tức mọi chữ trong đoạn đã hết, nên khe tại `i` **cho phép** cắt. Vậy **tập các khe cắt hợp lệ** = tập các `i` với `mx == i` (tập cắt "tối thiểu hóa" đoạn). Cắt ở **tất cả** các khe này cho số đoạn nhiều nhất, và đó là tập duy nhất đạt mức nhiều nhất (cắt thêm ở khe không cho phép là vi phạm).

**(c) Tính duy nhất và cấu trúc.** Một tập khe cắt hợp lệ bất kỳ là tập con của tập khe cho phép (do (b)); mọi tập con của tập các khe cho phép đều hợp lệ (gộp hai đoạn kề nhau không làm một chữ xuất hiện ở hai đoạn). Từ đó: số cách chia hợp lệ **với số đoạn bất kỳ** là `2^(m-1)` nếu có `m` đoạn nhiều nhất (đây là bài tập 5).

**(d) Các cách còn lại** đều đo cùng một thứ: khe `i` cắt được khi *không chữ nào bắc cầu qua nó*. Cách 3: khoảng gộp tách rời nhau đúng ở các khe không bị phủ. Cách 4: tập `opened` rỗng nghĩa là không chữ nào có lần xuất hiện cả trước lẫn sau khe. Cách 5: `cover[i] == 0` là đúng định nghĩa. Cách 1: `set(trước) & set(sau)` rỗng cũng là định nghĩa.

**Kiểm chứng bằng máy:** so cả 5 cách với oracle `2^(n-1)` trên **mọi** chuỗi trên bảng chữ cái `{a, b, c}` độ dài `<= 8`, rồi kiểm tra tính chất "tập khe cắt hợp lệ là tập con của các khe thuật toán cắt".

In [22]:
solutions = {
    "brute": partition_brute,
    "greedy": partition_greedy,
    "intervals": partition_intervals,
    "countdown": partition_countdown,
    "diff": partition_diff,
    "sol_greedy": sol_greedy,
    "sol_intervals": sol_intervals,
    "sol_countdown": sol_countdown,
    "sol_diff": sol_diff,
    "final": Solution().partitionLabels,
}

def all_strings(alphabet: str, max_len: int):
    out, frontier = [], [""]
    for _ in range(max_len):
        frontier = [x + c for x in frontier for c in alphabet]
        out += frontier
    return [x for x in out if x]

strings = all_strings("abc", 8)
for s in strings:
    ref = oracle(s)
    for name, fn in solutions.items():
        assert fn(s) == ref, (name, s, fn(s), ref)
print("so khớp oracle trên", len(strings), "chuỗi (a,b,c, độ dài 1..8) x", len(solutions), "cài đặt: OK")

# tính chất (c): mọi tập khe cắt hợp lệ là tập con của khe thuật toán cắt
def valid_cut_sets(s):
    n = len(s)
    for mask in range(1 << (n - 1)):
        cuts = {i for i in range(n - 1) if mask >> i & 1}
        bounds = [0] + [i + 1 for i in sorted(cuts)] + [n]
        parts = [s[bounds[k]:bounds[k + 1]] for k in range(len(bounds) - 1)]
        if sum(len(set(p)) for p in parts) == len(set(s)):
            yield cuts

for s in all_strings("abc", 7):
    algo_cuts = set(accumulate(partition_greedy(s)[:-1]))      # chỉ số kết thúc đoạn (1-based) = khe sau ký tự thứ k
    algo_gaps = {k - 1 for k in algo_cuts}
    valid_count = 0
    for cuts in valid_cut_sets(s):
        assert cuts <= algo_gaps, (s, cuts, algo_gaps)
        valid_count += 1
    assert valid_count == 2 ** len(algo_gaps), (s, valid_count)
print("tập khe cắt hợp lệ luôn là tập con của khe thuật toán cắt, và số cách chia hợp lệ = 2^(m-1): OK")

so khớp oracle trên 9840 chuỗi (a,b,c, độ dài 1..8) x 10 cài đặt: OK


tập khe cắt hợp lệ luôn là tập con của khe thuật toán cắt, và số cách chia hợp lệ = 2^(m-1): OK


## 11. Time / Space Complexity

Gọi `n = len(s)` (`<= 500`) và `|Σ| = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử từng đoạn bằng `set` | `O(n^2)` | `O(n)` |
| 2. Tham lam `last` | `O(n)` | `O(|Σ|)` |
| 3. Gộp khoảng | `O(n + |Σ| log |Σ|)` | `O(|Σ|)` |
| 4. Đếm ngược | `O(n)` | `O(|Σ|)` |
| 5. Mảng hiệu | `O(n)` | `O(n)` |
| Oracle thử mọi tập cắt | `O(2^n * n)` | `O(n)` |

Với `n = 500` mọi cách `O(n)` chạy dưới 1 ms; `O(n^2)` vài chục ms. Hãy đo:

In [23]:
random.seed(763)
tests = []
for alpha_size in (1, 2, 5, 26):
    alpha = "abcdefghijklmnopqrstuvwxyz"[:alpha_size]
    for _ in range(50):
        tests.append("".join(random.choice(alpha) for _ in range(500)))
# chuỗi nhiều đoạn: ghép các khối dùng bảng chữ cái rời nhau
tests.append("".join(chr(97 + i) * 19 for i in range(26)))
tests.append("abcdefghijklmnopqrstuvwxyz" * 19)

ref = None
for name in ["brute", "greedy", "intervals", "countdown", "diff"]:
    t0 = time.perf_counter()
    res = [solutions[name](s) for s in tests]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:10} {len(tests)} chuỗi dài ~500: {dt:8.2f} ms")
print("ví dụ chuỗi nhiều đoạn:", solutions["greedy"](tests[-2])[:5], "... tổng", len(solutions["greedy"](tests[-2])), "đoạn")

brute      202 chuỗi dài ~500:   500.09 ms
greedy     202 chuỗi dài ~500:    16.24 ms
intervals  202 chuỗi dài ~500:    10.11 ms
countdown  202 chuỗi dài ~500:    37.27 ms
diff       202 chuỗi dài ~500:    11.19 ms
ví dụ chuỗi nhiều đoạn: [19, 19, 19, 19, 19] ... tổng 26 đoạn


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với nhau và với oracle, và test ngẫu nhiên có seed cố định (đúng ràng buộc).

In [24]:
# 12.1 ví dụ đề
for s, expected in examples:
    for name, fn in solutions.items():
        assert fn(s) == expected, (name, s)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay)
edge = [
    ("a", [1]),                              # một ký tự
    ("aa", [2]),
    ("ab", [1, 1]),                          # mọi ký tự khác nhau -> cắt hết
    ("abc", [1, 1, 1]),
    ("aba", [3]),                            # a bắc cầu qua b
    ("abab", [4]),
    ("ababcc", [4, 2]),                      # ví dụ trong đề
    ("abaccb", [6]),                         # c nằm trong b ... b
    ("abcabc", [6]),
    ("aabbcc", [2, 2, 2]),
    ("caedbdedda", [1, 9]),                  # một đoạn 1 ký tự rồi đoạn dài
    ("a" * 500, [500]),
    ("".join(chr(97 + i) for i in range(26)), [1] * 26),
    ("".join(chr(97 + i) * 3 for i in range(26)), [3] * 26),
    ("abcdefghijklmnopqrstuvwxyz" * 2, [52]),
]
for s, expected in edge:
    if len(s) <= 12:
        assert oracle(s) == expected, ("oracle khác kỳ vọng", s)
    for name, fn in solutions.items():
        assert fn(s) == expected, (name, s, fn(s), expected)
    assert sum(expected) == len(s)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 phản ví dụ: các cách sai
def wrong_assign(s):          # mx = last[c] (ghi đè) thay vì max
    last = {c: i for i, c in enumerate(s)}
    mx = j = 0
    ans = []
    for i, c in enumerate(s):
        mx = last[c]
        if mx == i:
            ans.append(i - j + 1)
            j = i + 1
    return ans

def wrong_last_only(s):       # đóng đoạn khi i == last[c] của chính ký tự hiện tại
    last = {c: i for i, c in enumerate(s)}
    j = 0
    ans = []
    for i, c in enumerate(s):
        if last[c] == i:
            ans.append(i - j + 1)
            j = i + 1
    return ans

def wrong_first(s):           # dựng last bằng lần xuất hiện ĐẦU TIÊN
    last = {}
    for i, c in enumerate(s):
        last.setdefault(c, i)
    mx = j = 0
    ans = []
    for i, c in enumerate(s):
        mx = max(mx, last[c])
        if mx == i:
            ans.append(i - j + 1)
            j = i + 1
    return ans

def wrong_no_reset(s):        # quên đặt lại đầu đoạn j
    last = {c: i for i, c in enumerate(s)}
    mx = j = 0
    ans = []
    for i, c in enumerate(s):
        mx = max(mx, last[c])
        if mx == i:
            ans.append(i - j + 1)       # quên j = i + 1
    return ans

def wrong_off_by_one(s):      # độ dài thiếu 1
    last = {c: i for i, c in enumerate(s)}
    mx = j = 0
    ans = []
    for i, c in enumerate(s):
        mx = max(mx, last[c])
        if mx == i:
            ans.append(i - j)
            j = i + 1
    return ans

for name, fn, bad in [("mx = last[c]", wrong_assign, "abaccb"),
                      ("i == last[c]", wrong_last_only, "ababcc"),
                      ("last = lần đầu", wrong_first, "abab"),
                      ("quên đặt j", wrong_no_reset, "ababcc"),
                      ("lệch 1", wrong_off_by_one, "ababcc")]:
    truth = oracle(bad)
    print(f"{name:16} trên {bad!r:9}: sai = {fn(bad)}, đúng = {truth}")
    assert fn(bad) != truth
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

# 12.4 ngẫu nhiên có seed, đúng ràng buộc
rng = random.Random(2063)
for _ in range(1500):
    n = rng.randint(1, 500)
    alpha = "abcdefghijklmnopqrstuvwxyz"[:rng.randint(1, 26)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    res = {name: fn(s) for name, fn in solutions.items() if name != "brute" or n <= 120}
    vals = list(res.values())
    assert all(v == vals[0] for v in vals), (s, res)
    ans = vals[0]
    assert sum(ans) == n and all(a >= 1 for a in ans)
    # kiểm tra tính hợp lệ độc lập: mỗi chữ chỉ ở một đoạn
    pos, owner = 0, {}
    for k, size in enumerate(ans):
        for ch in s[pos:pos + size]:
            assert owner.setdefault(ch, k) == k, (s, ans)
        pos += size
print("12.4 ngẫu nhiên 1500 chuỗi (độ dài tới 500): OK")

# 12.5 ngẫu nhiên nhỏ so với oracle
rng = random.Random(5)
for _ in range(400):
    n = rng.randint(1, 12)
    alpha = "abcdefgh"[:rng.randint(1, 8)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    ref = oracle(s)
    for name, fn in solutions.items():
        assert fn(s) == ref, (name, s)
print("12.5 ngẫu nhiên nhỏ so với oracle: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 15 ca
mx = last[c]     trên 'abaccb' : sai = [3, 2, 1], đúng = [6]
i == last[c]     trên 'ababcc' : sai = [3, 1, 2], đúng = [4, 2]
last = lần đầu   trên 'abab'   : sai = [1, 1], đúng = [4]
quên đặt j       trên 'ababcc' : sai = [4, 6], đúng = [4, 2]
lệch 1           trên 'ababcc' : sai = [3, 1], đúng = [4, 2]
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


12.4 ngẫu nhiên 1500 chuỗi (độ dài tới 500): OK


12.5 ngẫu nhiên nhỏ so với oracle: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gán `mx = last[c]` thay vì `max`

Mỗi chữ có thể kéo `mx` tới chỗ khác nhau; nếu **ghi đè** thì một chữ có `last` gần xóa mất yêu cầu xa hơn của chữ trước.

In [25]:
s = "abaccb"
print("sai :", wrong_assign(s))
print("đúng:", partition_greedy(s), "(b xuất hiện ở 1 và 5, nên cả chuỗi là một đoạn)")
assert wrong_assign(s) != partition_greedy(s) == [6]

sai : [3, 2, 1]
đúng: [6] (b xuất hiện ở 1 và 5, nên cả chuỗi là một đoạn)


Vì sao sai: ở `i = 2` (`a`, `last = 2`) ta ghi đè `mx = 2` và quên mất `b` đã đòi tới 5. **Đúng:** `mx = max(mx, last[c])`.

### Lỗi 2 — Cắt khi `i == last[c]` (chỉ nhìn ký tự hiện tại)

Chữ cuối cùng của đoạn chưa chắc là chữ có `last` xa nhất.

In [26]:
s = "ababcc"
print("sai :", wrong_last_only(s))
print("đúng:", partition_greedy(s))
assert wrong_last_only(s) != partition_greedy(s)

sai : [3, 1, 2]
đúng: [4, 2]


Vì sao sai: tại `i = 2` (`a`, `last[a] = 2`) ta cắt, nhưng `b` còn xuất hiện ở 3. **Đúng:** so với `mx`, tức **tối đa** của `last` mọi chữ đã vào đoạn.

### Lỗi 3 — Dựng `last` bằng lần xuất hiện đầu tiên

Dùng `setdefault` (hoặc `if c not in last`) cho `last` thì ta đang lưu **first**, không phải last.

In [27]:
s = "abab"
print("sai :", wrong_first(s))
print("đúng:", partition_greedy(s))
assert wrong_first(s) == [1, 1] and partition_greedy(s) == [4]

sai : [1, 1]
đúng: [4]


Vì sao sai: `last` phải bị **ghi đè** ở mỗi lần gặp (`{c: i for ...}` làm đúng điều đó), để cuối cùng giữ chỉ số lớn nhất. **Đúng:** `last = {c: i for i, c in enumerate(s)}`.

### Lỗi 4 — Quên đặt lại đầu đoạn

Nếu không cập nhật `j = i + 1` sau mỗi lần cắt thì độ dài tính từ đầu chuỗi (cộng dồn).

In [28]:
s = "ababcc"
print("sai :", wrong_no_reset(s), "(cộng dồn: 4, 6)")
print("đúng:", partition_greedy(s))
assert wrong_no_reset(s) == [4, 6] and partition_greedy(s) == [4, 2]

sai : [4, 6] (cộng dồn: 4, 6)
đúng: [4, 2]


Vì sao sai: độ dài đoạn là `i - j + 1` với `j` **đầu đoạn hiện tại**. **Đúng:** sau khi thêm vào kết quả, đặt `j = i + 1`.

### Lỗi 5 — Lệch 1 khi tính độ dài

Đoạn `[j..i]` (cả hai đầu mút bao gồm) có `i - j + 1` phần tử, không phải `i - j`.

In [29]:
s = "ababcc"
print("sai :", wrong_off_by_one(s))
print("đúng:", partition_greedy(s))
assert wrong_off_by_one(s) == [3, 1] and partition_greedy(s) == [4, 2]

sai : [3, 1]
đúng: [4, 2]


Vì sao sai: đoạn `"a"` (chỉ số 0..0) có độ dài 1 = `0 - 0 + 1`. **Đúng:** luôn `i - j + 1`, hoặc dùng `s[j:i+1]` rồi `len`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "cắt thành nhiều phần nhất", "mỗi phần tử chỉ thuộc một nhóm", ràng buộc dạng "trong cùng một đoạn" -> **khoảng `[first, last]` + gộp** hoặc **tham lam với vị trí cuối**.
- **Bản đồ lan truyền:** mở rộng đoạn khi gặp phần tử đòi xa hơn (kiểu Jump Game: `farthest = max(farthest, ...)`; cắt khi `i == farthest`).
- Tham lam ở đây đúng vì **cắt càng sớm càng tốt** và việc cắt sớm không làm hỏng các cắt sau (các khe cho phép **độc lập** nhau).
- **Merge Intervals ẩn:** nhìn mỗi ký tự như một khoảng rồi gộp.
- Mảng hiệu `diff` là công cụ chung cho "đánh dấu phủ lên nhiều khoảng" rồi đọc bằng tổng tích lũy.
- Bảng chữ cái nhỏ (26) cho phép bộ nhớ `O(1)`.
- Test nhanh nên có: một ký tự, mọi ký tự khác nhau, mọi ký tự giống nhau, `"abab..."`, một chữ bắc cầu cả chuỗi.

## 15. Biến thể

1. **Trả về các đoạn là chuỗi** thay vì độ dài.
2. **Chỉ trả số đoạn** (số khe cắt + 1).
3. **Số cách chia hợp lệ với số đoạn bất kỳ**: `2^(m-1)` nếu có `m` đoạn nhiều nhất (bài tập 5; tương tự LeetCode 2963 Count the Number of Good Partitions).
4. **Mảng số nguyên** thay vì chuỗi: cùng thuật toán với `last` là dict theo giá trị.
5. Bài liên quan: Merge Intervals (56), Jump Game II (45), Partition Array into Disjoint Intervals (915).

Dưới đây là code cho biến thể 1, 2 và 4, kiểm bằng oracle.

In [30]:
def partition_strings(s: str) -> List[str]:
    last = {c: i for i, c in enumerate(s)}
    mx = j = 0
    parts = []
    for i, c in enumerate(s):
        mx = max(mx, last[c])
        if mx == i:
            parts.append(s[j:i + 1])
            j = i + 1
    return parts

def count_parts(s: str) -> int:
    return len(partition_greedy(s))

def partition_array(nums: List[int]) -> List[int]:
    last = {v: i for i, v in enumerate(nums)}
    mx = j = 0
    ans = []
    for i, v in enumerate(nums):
        mx = max(mx, last[v])
        if mx == i:
            ans.append(i - j + 1)
            j = i + 1
    return ans

print(partition_strings("ababcbacadefegdehijhklij"))
print(count_parts("eccbbbbdec"), partition_array([1, 2, 1, 3, 4, 3, 5]))
assert partition_strings("ababcc") == ["abab", "cc"]
assert count_parts("abc") == 3
assert partition_array([1, 2, 1, 3, 4, 3, 5]) == [3, 3, 1]
for s in all_strings("abc", 6):
    assert partition_array([ord(c) for c in s]) == partition_greedy(s)
    assert "".join(partition_strings(s)) == s
    assert count_parts(s) == len(oracle(s))
print("Biến thể: OK")

['ababcbaca', 'defegde', 'hijhklij']
1 [3, 3, 1]
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng đoạn | `set(trong) & set(ngoài)` rỗng | `O(n^2)` | `O(n)` | Hiểu đề; chuỗi ngắn |
| 2. Tham lam `last` | Đường chân trời `mx` | `O(n)` | `O(26)` | **Cách nộp chính**: ngắn nhất, nhanh nhất |
| 3. Gộp khoảng | `[first, last]` + Merge Intervals | `O(n + k log k)` | `O(26)` | Khi bài mở rộng sang khoảng tổng quát |
| 4. Đếm ngược | `remaining[c]` + tập `opened` | `O(n)` | `O(26)` | Khi chỉ biết số lần xuất hiện, không cần vị trí |
| 5. Mảng hiệu | `cover[i] == 0` ở khe | `O(n)` | `O(n)` | Khi muốn xem bài toán như "phủ khoảng" |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print({c: i for i, c in enumerate("abca")})
print(partition_greedy("abca"))
print(partition_greedy("abcd"))
```

In [31]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `s = "caedbdedda"`: ghi lại bảng `last` (dạng dict), và danh sách độ dài các đoạn.

In [32]:
# Bài 2: điền kết quả tính tay
hand_last = None          # dict ký tự -> chỉ số cuối
hand_answer = None        # list độ dài các đoạn

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 2.

In [33]:
def exercise_3_greedy(s: str) -> List[int]:
    # TODO 1: dựng last: ký tự -> chỉ số xuất hiện cuối cùng
    mx = j = 0
    ans = []
    for i, c in enumerate(s):
        # TODO 2: cập nhật mx bằng max
        # TODO 3: nếu mx == i thì thêm độ dài đoạn vào ans và đặt lại j
        pass
    return ans


# assert exercise_3_greedy("ababcbacadefegdehijhklij") == [9, 7, 8]
# assert exercise_3_greedy("eccbbbbdec") == [10]
# assert exercise_3_greedy("abc") == [1, 1, 1]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_intervals(s)`: dựng khoảng `[first, last]` cho mỗi chữ, sắp xếp, gộp các khoảng giao nhau và trả về độ dài các khoảng gộp.

In [34]:
def exercise_4_intervals(s: str) -> List[int]:
    # TODO
    pass


# assert exercise_4_intervals("ababcbacadefegdehijhklij") == [9, 7, 8]
# assert exercise_4_intervals("eccbbbbdec") == [10]
# assert exercise_4_intervals("a") == [1]

### Bài 5 — Mini challenge: đếm số cách chia hợp lệ (khó)

Chia `s` thành **một số đoạn bất kỳ** (không nhất thiết nhiều nhất) sao cho mỗi chữ cái chỉ xuất hiện trong một đoạn. Đếm có bao nhiêu cách chia (hai cách khác nhau nếu tập điểm cắt khác nhau). Chuỗi không cắt (một đoạn) cũng tính.

```python
exercise_5_count_partitions("ababcc")        -> 2   ("ababcc" ; "abab|cc")
exercise_5_count_partitions("abc")           -> 4   (2^(3-1))
exercise_5_count_partitions("aba")           -> 1
exercise_5_count_partitions("eccbbbbdec")    -> 1
```
Gợi ý: gọi `m` là số đoạn nhiều nhất; mỗi khe trong `m - 1` khe cho phép có thể cắt hoặc không.

In [35]:
def exercise_5_count_partitions(s: str) -> int:
    # TODO
    pass


# assert exercise_5_count_partitions("ababcc") == 2
# assert exercise_5_count_partitions("abc") == 4
# assert exercise_5_count_partitions("aba") == 1
# assert exercise_5_count_partitions("eccbbbbdec") == 1

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: dict comprehension: khóa trùng thì giá trị sau ghi đè giá trị trước.
- Mức 2: `"abca"`: `a` xuất hiện ở 0 và 3, nên `last["a"] = 3`.
- Mức 3: `{'a': 3, 'b': 1, 'c': 2}`, `[4]` (a bắc cầu cả chuỗi), `[1, 1, 1, 1]`.

**Bài 2**
- Mức 1: liệt kê vị trí cuối từng chữ cái: `c, a, e, d, b`.
- Mức 2: `c` -> 0, `a` -> 9, `e` -> 6, `d` -> 8, `b` -> 4... kiểm tra lại bằng code `{c: i ...}`.
- Mức 3: `{'c': 0, 'a': 9, 'e': 6, 'd': 8, 'b': 4}`; đoạn 1 là `"c"` (`mx = 0` tại `i = 0`), đoạn 2 phủ `a` tới 9: `[1, 9]`.

**Bài 3**
- Mức 1: `last = {c: i for i, c in enumerate(s)}`.
- Mức 2: `mx = max(mx, last[c])`.
- Mức 3: `if mx == i: ans.append(i - j + 1); j = i + 1`.

**Bài 4**
- Mức 1: `first.setdefault(c, i)` và `last[c] = i`.
- Mức 2: `sorted((first[c], last[c]) for c in first)`.
- Mức 3: gộp nếu `a <= merged[-1][1]` (`max` hai đầu mút phải); trả `[b - a + 1 for a, b in merged]`.

**Bài 5**
- Mức 1: tìm đáp án Bài toán gốc `m = len(partition_greedy(s))`.
- Mức 2: mỗi khe trong `m - 1` khe cho phép được cắt hoặc không (hai lựa chọn độc lập).
- Mức 3: `return 2 ** (m - 1)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [36]:
print({c: i for i, c in enumerate("abca")})
print(partition_greedy("abca"))
print(partition_greedy("abcd"))

assert {c: i for i, c in enumerate("abca")} == {"a": 3, "b": 1, "c": 2}
assert partition_greedy("abca") == [4]
assert partition_greedy("abcd") == [1, 1, 1, 1]

{'a': 3, 'b': 1, 'c': 2}
[4]
[1, 1, 1, 1]


### Đáp án Bài 2

`last = {'c': 0, 'a': 9, 'e': 6, 'd': 8, 'b': 4}`. Đoạn đầu: `i = 0`, `c`, `mx = 0` -> cắt, độ dài 1. Đoạn sau bắt đầu ở 1: `a` đòi tới 9 nên kéo tới hết chuỗi -> độ dài 9.

In [37]:
hand_last_solution = {"c": 0, "a": 9, "e": 6, "d": 8, "b": 4}
hand_answer_solution = [1, 9]
assert {c: i for i, c in enumerate("caedbdedda")} == hand_last_solution
assert partition_greedy("caedbdedda") == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [38]:
def exercise_3_greedy_solution(s: str) -> List[int]:
    last = {c: i for i, c in enumerate(s)}
    mx = j = 0
    ans = []
    for i, c in enumerate(s):
        mx = max(mx, last[c])
        if mx == i:
            ans.append(i - j + 1)
            j = i + 1
    return ans

assert exercise_3_greedy_solution("ababcbacadefegdehijhklij") == [9, 7, 8]
assert exercise_3_greedy_solution("eccbbbbdec") == [10]
assert exercise_3_greedy_solution("abc") == [1, 1, 1]
for s in all_strings("abc", 7):
    assert exercise_3_greedy_solution(s) == oracle(s), s
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [39]:
def exercise_4_intervals_solution(s: str) -> List[int]:
    first, last = {}, {}
    for i, c in enumerate(s):
        first.setdefault(c, i)
        last[c] = i
    merged = []
    for a, b in sorted((first[c], last[c]) for c in first):
        if merged and a <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    return [b - a + 1 for a, b in merged]

assert exercise_4_intervals_solution("ababcbacadefegdehijhklij") == [9, 7, 8]
assert exercise_4_intervals_solution("eccbbbbdec") == [10]
assert exercise_4_intervals_solution("a") == [1]
for s in all_strings("abc", 7):
    assert exercise_4_intervals_solution(s) == oracle(s), s
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [40]:
def exercise_5_count_partitions_solution(s: str) -> int:
    m = len(partition_greedy(s))
    return 2 ** (m - 1)

def count_oracle(s: str) -> int:
    return sum(1 for _ in valid_cut_sets(s))

assert exercise_5_count_partitions_solution("ababcc") == 2
assert exercise_5_count_partitions_solution("abc") == 4
assert exercise_5_count_partitions_solution("aba") == 1
assert exercise_5_count_partitions_solution("eccbbbbdec") == 1
for s in all_strings("abc", 8):
    assert exercise_5_count_partitions_solution(s) == count_oracle(s), s
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tham lam với vị trí cuối)

```python
class Solution:
    def partitionLabels(self, s: str) -> List[int]:
        last = {c: i for i, c in enumerate(s)}   # vị trí xuất hiện CUỐI của mỗi ký tự
        mx = j = 0                                # mx: tới đâu đoạn bắt buộc phải vươn; j: đầu đoạn
        ans = []
        for i, c in enumerate(s):
            mx = max(mx, last[c])
            if mx == i:                           # mọi ký tự trong đoạn đã hết -> cắt
                ans.append(i - j + 1)
                j = i + 1
        return ans
```

Công thức nhớ nhanh: cắt tại `i` khi `i == max(last[c] của mọi c đã duyệt trong đoạn)`.

### Checklist trước khi submit

- [ ] `last` dựng bằng **ghi đè** (vị trí cuối), không phải `setdefault` (vị trí đầu)?
- [ ] `mx = max(mx, last[c])`, không phải gán thẳng?
- [ ] Cắt khi `i == mx` (không phải `i == last[c]`)?
- [ ] Độ dài `i - j + 1` và đặt lại `j = i + 1` sau mỗi lần cắt?
- [ ] Trả về **list độ dài** (không phải list chuỗi)?
- [ ] Đã thử `"a"`, `"abc"`, `"aaaa"`, `"abab"`, `"eccbbbbdec"`?
- [ ] Tên method đúng `partitionLabels(self, s)`?